# Getting Started With The Hybrid Numba CUDA Sweep Backend

This notebook demonstrates the `NbHybridCUDASweepBackend`, which runs a parameter sweep by assigning one independent hybrid simulation to each CUDA thread. It is useful when many simulations share the same network topology but differ by one or more model or coupling parameters.

The example below builds a small hybrid `NetworkSet`, sweeps a coupling-function parameter on the GPU, and compares one sweep point against the CPU Numba hybrid backend.

## Requirements

You need a CUDA-capable GPU, a working NVIDIA driver, and `numba.cuda`. The notebook checks availability before launching kernels. If CUDA is not available, the setup cells still show the expected API, but the GPU execution cell is skipped.

In [ ]:
import os
import tempfile

os.environ.setdefault("TVB_USER_HOME", os.path.join(tempfile.gettempdir(), "tvb-user"))
os.environ.setdefault("MPLCONFIGDIR", os.path.join(tempfile.gettempdir(), "matplotlib"))

import numpy as np
import scipy.sparse as sp

try:
    import numba.cuda
    CUDA_AVAILABLE = numba.cuda.is_available()
except Exception as exc:
    CUDA_AVAILABLE = False
    CUDA_ERROR = exc
else:
    CUDA_ERROR = None

CUDA_AVAILABLE

## Imports

`NbHybridCUDASweepBackend` reuses the hybrid Numba backend's network analysis, but compiles a separate `numba.cuda` sweep kernel.

In [ ]:
from tvb.simulator.backend.nb_hybrid_cuda_sweep_backend import NbHybridCUDASweepBackend
from tvb.simulator.backend.nb_hybrid import NbHybridBackend
from tvb.simulator.hybrid.network import NetworkSet
from tvb.simulator.hybrid.subnetwork import Subnetwork
from tvb.simulator.hybrid.intra_projection import IntraProjection
from tvb.simulator.hybrid.coupling import Scaling
from tvb.simulator.integrators import HeunDeterministic
from tvb.simulator.models.oscillator import Generic2dOscillator

## Build A Small Hybrid Network

This uses one `Generic2dOscillator` subnetwork with an intra-projection. The CUDA backend expects sparse projection data, so the weights and tract lengths are CSR matrices.

In [ ]:
DT = 0.01
N_NODES = 8


def build_g2d_network(n_nodes=N_NODES):
    weights = sp.csr_matrix(np.eye(n_nodes, dtype=np.float64) * 0.1)
    lengths = sp.csr_matrix(np.zeros((n_nodes, n_nodes), dtype=np.float64))

    model = Generic2dOscillator()
    model.configure()

    subnet = Subnetwork(
        name="g2d",
        model=model,
        scheme=HeunDeterministic(dt=DT),
        nnodes=n_nodes,
    )
    subnet.configure()

    intra = IntraProjection(
        source_cvar=np.array([0], dtype=np.int32),
        target_cvar=np.array([0], dtype=np.int32),
        weights=weights,
        lengths=lengths,
        cv=1.0,
        dt=DT,
        scale=1.0,
        cfun=Scaling(a=np.array([1.0])),
    )
    subnet.projections = [intra]
    subnet.configure()

    network = NetworkSet(subnets=[subnet], projections=[])
    network.configure()
    return network


network = build_g2d_network()
network.subnets[0].name, network.subnets[0].nnodes, len(network.subnets[0].projections)

## Define Sweep Values

Without an explicit `sweep_descriptor`, the backend sweeps the first parameter of the first projection coupling function. Here that is `Scaling.a` on the intra-projection.

In [ ]:
nstep = 20
sweep_values = np.array([[0.5], [1.0], [1.5], [2.0]], dtype=np.float32)
initial_state = np.zeros((2, N_NODES, 1), dtype=np.float64)

sweep_values

## Run The CUDA Sweep

The output `tavg` is a list with one array per subnetwork. For this network there is one subnetwork, so `result["tavg"][0]` has shape `(n_sweeps, n_voi, n_nodes, n_modes)`.

In [ ]:
if not CUDA_AVAILABLE:
    print("CUDA is not available; skipping GPU execution.")
    if CUDA_ERROR is not None:
        print(repr(CUDA_ERROR))
    gpu_result = None
else:
    gpu_result = NbHybridCUDASweepBackend().run_sweep(
        network,
        sweep_values=sweep_values,
        nstep=nstep,
        initial_states=[initial_state],
        verbose=True,
    )
    print("result keys:", sorted(gpu_result.keys()))
    print("tavg[0] shape:", gpu_result["tavg"][0].shape)

## Compare One Sweep Point With CPU Numba

The CPU `NbHybridBackend` does not execute all sweep points in one CUDA-style kernel. For validation, we run the same network with the same initial state and compare the temporal average for the sweep value `1.0`.

In [ ]:
def cpu_tavg_for_current_network(network_set, nstep, initial_state):
    results = NbHybridBackend().run_network(
        network_set,
        nstep=nstep,
        chunk_size=1,
        initial_states=[initial_state],
    )
    # results are per-subnet tuples: (times, data, ctavg)
    return results[0][1].mean(axis=0).astype(np.float32)


if gpu_result is None:
    print("Skipped because CUDA result is unavailable.")
else:
    cpu_tavg = cpu_tavg_for_current_network(network, nstep, initial_state)
    gpu_tavg = gpu_result["tavg"][0][1]  # sweep value 1.0
    max_error = np.max(np.abs(gpu_tavg - cpu_tavg))
    print("max abs error:", max_error)
    np.testing.assert_allclose(gpu_tavg, cpu_tavg, rtol=1e-3, atol=1e-4)

## Chunking And Batching

`chunk_size` splits the time loop into multiple kernel launches. `max_batch_sweeps` limits how many sweep points are resident on the GPU at once. These options are useful for larger sweeps or larger networks.

In [ ]:
if gpu_result is None:
    print("Skipped because CUDA result is unavailable.")
else:
    chunked = NbHybridCUDASweepBackend().run_sweep(
        network,
        sweep_values=sweep_values,
        nstep=100,
        initial_states=[initial_state],
        chunk_size=25,
        max_batch_sweeps=2,
    )
    print("chunked tavg shape:", chunked["tavg"][0].shape)
    print("snapshot step offset:", chunked["snapshot"]["step_offset"])

## Notes

- The CUDA sweep backend parallelizes across sweep points. Each GPU thread runs a full simulation for one row of `sweep_values`.
- The backend reuses `NbHybridBackend` for compatibility checks and network analysis, then generates a separate `numba.cuda` kernel.
- Outputs are `float32` because the generated CUDA path stores runtime arrays in single precision.
- For more coverage examples, see `tvb_library/tvb/tests/library/simulator/backend/test_nb_hybrid_cuda_sweep.py`.